# 02 · Initial unweighted U-Net baseline

Historical experiment: train RGB and RGB+NIR U-Nets on Sen1Floods11 with unweighted cross-entropy. The cloud run was completed; the saved RGB checkpoint predicted no water, motivating the diagnostics in 03–04. This local copy retains the executable experiment; final project results use Notebook 08.

Run training on the CUDA environment with the project dependencies installed. Each model trains for 50 epochs.

## 1. Configuration

| Setting | Value |
|---|---|
| Inputs | RGB: B4/B3/B2; RGB+NIR: B4/B3/B2/B8 |
| Split | Train 252 / validation 89 / test 90 / Bolivia 15 |
| Budget | 50 epochs per model; batch size 8 |
| Training samples | One random 256 × 256 crop per training image per epoch |
| Augmentation | Synchronized horizontal and vertical flips, each with probability 0.5 |
| Optimizer | Adam, learning rate 1e-4, weight decay 0; no scheduler |
| Runtime | CUDA AMP; seed 42; num_workers=0 |
| Evaluation | Full 512 × 512 images, batch size 1 |
| Selection | Highest validation image-mean water IoU; ties retain the earlier epoch |

These were shared baseline settings. The configuration cell creates a new run directory; the training cell starts both experiments. Checkpoints do not contain optimizer state for resuming training.

In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import json
import time
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from IPython.display import display, Markdown
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
BASE = ROOT / "data/sen1floods11/v1.1"
HAND = BASE / "data/flood_events/HandLabeled"
CFG = dict(epochs=50, batch_size=8, crop_size=256, lr=1e-4,
           weight_decay=0.0, seed=42, num_workers=0, amp=True)
EXPERIMENTS = {"rgb": ["B4", "B3", "B2"],
               "rgb_nir": ["B4", "B3", "B2", "B8"]}
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)
USE_AMP = CFG["amp"] and device.type == "cuda"
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
RUN = ROOT / "runs" / datetime.now().strftime("rgb_nir_%Y%m%d_%H%M%S_%f")
RUN.mkdir(parents=True)
(RUN / "config.json").write_text(json.dumps({"training": CFG, "bands": EXPERIMENTS,
    "torch": str(torch.__version__), "device": str(device)}, indent=2), encoding="utf-8")
print("Results:", RUN)

splits = {}
for split in ["train", "valid", "test", "bolivia"]:
    with (BASE / f"splits/flood_handlabeled/flood_{split}_data.csv").open(newline="") as f:
        splits[split] = list(csv.reader(f))
print("Official split sizes:", {key: len(rows) for key, rows in splits.items()})


## 2. Shared data and sampling

Use the official split lists, replacing `_S1Hand` with `_S2Hand` to locate optical images. Read B4/B3/B2/B8, divide DN by 10000, and apply one shared valid-pixel mask before selecting three or four channels. Labels are 0 (non-water), 1 (water), and -1 (ignored).

Crop locations, flips and sample order are synchronized between models. Validation and testing use full images without augmentation.

In [ ]:
class FloodDataset(Dataset):
    def __init__(self, rows, channels, training=False):
        self.rows = rows
        self.channels = channels
        self.training = training
        self.epoch = 0

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        image_name, label_name = self.rows[index]
        image_name = image_name.replace("_S1Hand", "_S2Hand")
        with rasterio.open(HAND / "S2Hand" / image_name) as src:
            bands = [src.descriptions.index(b) + 1 for b in ["B4", "B3", "B2", "B8"]]
            x = src.read(bands, masked=True).astype("float32").filled(np.nan)
        with rasterio.open(HAND / "LabelHand" / label_name) as src:
            y = src.read(1).astype("int64")
        y[~np.isfinite(x).all(axis=0)] = -1
        x = np.nan_to_num(x / 10000.0)
        if self.training:
            rng = np.random.default_rng([CFG["seed"], self.epoch, index])
            size = CFG["crop_size"]
            top = int(rng.integers(0, y.shape[0] - size + 1))
            left = int(rng.integers(0, y.shape[1] - size + 1))
            x = x[:, top:top+size, left:left+size]
            y = y[top:top+size, left:left+size]
            for axis in [0, 1]:
                if rng.random() < 0.5:
                    x = np.flip(x, axis=axis + 1)
                    y = np.flip(y, axis=axis)
        return torch.from_numpy(x[:self.channels].copy()), torch.from_numpy(y.copy())

example = FloodDataset(splits["train"], channels=4, training=True)
sample_x, sample_y = example[0]
print("Training sample:", tuple(sample_x.shape), "target:", tuple(sample_y.shape))
print("Shared valid pixels:", int((sample_y != -1).sum()))


## 3. U-Net and initialization

The earlier U-Net uses encoder widths 64/128/256/512, a 1024-channel bottleneck and a symmetric decoder with transposed convolutions. Each block has two padded 3 × 3 convolutions and ReLU; a 1 × 1 head produces two logits per pixel.

Both models start from the same four-channel reference initialization. RGB uses the first three input-channel weights; all matching layers share their initial values. Each model is then trained independently.

In [ ]:
import torch
from torch import nn

class DoubleConv(nn.Sequential):
    def __init__(self, in_channels, out_channels):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )

class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()
        self.encoders = nn.ModuleList([
            DoubleConv(in_channels, 64), DoubleConv(64, 128),
            DoubleConv(128, 256), DoubleConv(256, 512),
        ])
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)
        self.upsamples = nn.ModuleList([
            nn.ConvTranspose2d(1024, 512, 2, stride=2),
            nn.ConvTranspose2d(512, 256, 2, stride=2),
            nn.ConvTranspose2d(256, 128, 2, stride=2),
            nn.ConvTranspose2d(128, 64, 2, stride=2),
        ])
        self.decoders = nn.ModuleList([
            DoubleConv(1024, 512), DoubleConv(512, 256),
            DoubleConv(256, 128), DoubleConv(128, 64),
        ])
        self.head = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        skips = []
        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        for upsample, decoder, skip in zip(self.upsamples, self.decoders, reversed(skips)):
            x = upsample(x)
            x = decoder(torch.cat([skip, x], dim=1))
        return self.head(x)


torch.manual_seed(CFG["seed"])
reference = UNet(in_channels=4, num_classes=2)
initial_state = {key: value.detach().clone() for key, value in reference.state_dict().items()}
del reference

def create_model(channels):
    model = UNet(in_channels=channels, num_classes=2)
    state = dict(initial_state)
    state["encoders.0.0.weight"] = initial_state["encoders.0.0.weight"][:, :channels].clone()
    model.load_state_dict(state)
    return model.to(device)


## 4. Loss and metrics

Cross-entropy is averaged over valid pixels; labels -1 are ignored and all-ignored batches are skipped.

Water IoU = TP/(TP+FP+FN), precision = TP/(TP+FP), recall = TP/(TP+FN), and F1 = 2TP/(2TP+FP+FN). Global IoU pools valid pixels; mean IoU averages image-level water IoUs, excluding images with zero water union. Undefined ratios remain NaN. Evaluation metrics are unweighted.

In [ ]:
def divide(numerator, denominator):
    return float(numerator / denominator) if denominator else float("nan")

class WaterMetrics:
    def __init__(self):
        self.matrix = np.zeros((2, 2), dtype=np.int64)
        self.image_ious = []
        self.valid_images = 0

    def update(self, prediction, target):
        for pred, truth in zip(prediction.cpu(), target.cpu()):
            valid = truth != -1
            if not valid.any():
                continue
            matrix = torch.bincount(2 * truth[valid] + pred[valid], minlength=4).reshape(2, 2).numpy()
            self.matrix += matrix
            self.valid_images += 1
            tn, fp, fn, tp = matrix.ravel()
            union = tp + fp + fn
            if union:
                self.image_ious.append(float(tp / union))

    def compute(self):
        tn, fp, fn, tp = self.matrix.ravel()
        return dict(global_water_iou=divide(tp, tp + fp + fn),
                    mean_water_iou=float(np.mean(self.image_ious)) if self.image_ious else float("nan"),
                    precision=divide(tp, tp + fp), recall=divide(tp, tp + fn),
                    f1=divide(2 * tp, 2 * tp + fp + fn),
                    valid_pixels=int(self.matrix.sum()), valid_images=self.valid_images,
                    images_with_water_union=len(self.image_ious))

def run_epoch(model, loader, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    metrics = WaterMetrics()
    loss_sum, pixel_count, skipped = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        count = int((y != -1).sum())
        if count == 0:
            skipped += 1
            continue
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(x)
                loss = F.cross_entropy(logits, y, ignore_index=-1, reduction="sum") / count
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        loss_sum += loss.detach().item() * count
        pixel_count += count
        metrics.update(logits.detach().argmax(dim=1), y)
    result = metrics.compute()
    result.update(loss=divide(loss_sum, pixel_count), skipped_batches=skipped)
    return result


## 5. Training and checkpointing

Evaluate after each epoch and save `best.pt` when validation image-mean water IoU improves. `history.csv` records loss, IoU and elapsed time. This original experiment saves the best checkpoint only; it does not retain the final epoch if that epoch is not best.

In [ ]:
def write_csv(path, rows):
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def train_experiment(name, bands):
    out = RUN / name
    out.mkdir(exist_ok=True)
    dataset = FloodDataset(splits["train"], len(bands), training=True)
    train_loader = DataLoader(dataset, batch_size=CFG["batch_size"], shuffle=True,
        num_workers=CFG["num_workers"], generator=torch.Generator().manual_seed(CFG["seed"]))
    valid_loader = DataLoader(FloodDataset(splits["valid"], len(bands)), batch_size=1,
        num_workers=CFG["num_workers"])
    model = create_model(len(bands))
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    best_iou, history = -1.0, []
    for epoch in range(1, CFG["epochs"] + 1):
        start = time.perf_counter()
        dataset.epoch = epoch
        train = run_epoch(model, train_loader, optimizer, scaler)
        valid = run_epoch(model, valid_loader)
        row = {"epoch": epoch, "seconds": time.perf_counter() - start,
               **{"train_" + k: v for k, v in train.items()},
               **{"valid_" + k: v for k, v in valid.items()}}
        history.append(row)
        write_csv(out / "history.csv", history)
        if valid["mean_water_iou"] > best_iou:
            best_iou = valid["mean_water_iou"]
            torch.save({"model": model.state_dict(), "bands": bands, "config": CFG,
                        "epoch": epoch, "validation": valid}, out / "best.pt")
        print(f"{name} | {epoch:02d}/{CFG['epochs']} | train loss {train['loss']:.4f} | "
              f"val loss {valid['loss']:.4f} | val mean IoU {valid['mean_water_iou']:.4f} | "
              f"{row['seconds']:.0f}s", flush=True)
    del model, optimizer, scaler
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return history


## 6. Train both models

The following cell starts RGB, followed by RGB+NIR. Start from the configuration cell for a new run; repeating training within the same run replaces its files.

In [ ]:
histories = {}
for name, bands in EXPERIMENTS.items():
    histories[name] = train_experiment(name, bands)
print("Training finished:", RUN)


## 7. Learning curves

Solid and dashed lines show training and validation loss. The second panel shows validation image-mean water IoU. Values are read directly from this run's epoch logs.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for (name, history), color in zip(histories.items(), ["#167caf", "#df893b"]):
    epochs = [row["epoch"] for row in history]
    axes[0].plot(epochs, [row["train_loss"] for row in history], color=color, label=name + " train")
    axes[0].plot(epochs, [row["valid_loss"] for row in history], color=color, linestyle="--", label=name + " valid")
    axes[1].plot(epochs, [row["valid_mean_water_iou"] for row in history], color=color, label=name)
axes[0].set(xlabel="Epoch", ylabel="Cross entropy", title="Training and validation loss")
axes[1].set(xlabel="Epoch", ylabel="Image-mean water IoU", title="Validation · model selection", ylim=(0, 1))
for ax in axes:
    ax.legend(); ax.grid(alpha=0.2)
fig.savefig(RUN / "learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)


## 8. Test evaluation

Load each validation-selected `best.pt` and evaluate the 90 test images and 15 Bolivia images separately. Save `test_metrics.csv`. Predictions use argmax, with ties assigned to non-water; test scores do not select checkpoints.

In [ ]:
def load_best(name):
    checkpoint = torch.load(RUN / name / "best.pt", map_location="cpu", weights_only=True)
    model = UNet(in_channels=len(checkpoint["bands"]), num_classes=2)
    model.load_state_dict(checkpoint["model"])
    return model.to(device).eval(), checkpoint

results = []
for name, bands in EXPERIMENTS.items():
    model, checkpoint = load_best(name)
    for split in ["test", "bolivia"]:
        loader = DataLoader(FloodDataset(splits[split], len(bands)), batch_size=1,
                            num_workers=CFG["num_workers"])
        metrics = run_epoch(model, loader)
        results.append({"experiment": name, "split": split,
                        "best_epoch": checkpoint["epoch"], **metrics})
    del model, checkpoint
write_csv(RUN / "test_metrics.csv", results)
columns = ["experiment", "split", "best_epoch", "mean_water_iou", "global_water_iou", "f1", "precision", "recall"]
lines = ["| " + " | ".join(columns) + " |", "|" + "---|" * len(columns)]
for row in results:
    lines.append("| " + " | ".join(f"{row[k]:.4f}" if isinstance(row[k], float) else str(row[k]) for k in columns) + " |")
display(Markdown("\n".join(lines)))


## 9. Prediction example

Compare both models on the first image in the official test list (`SHOW_INDEX=0`). Display RGB, NIR, human labels and both predictions on the same grid. RGB stretching is for display only; NIR uses a fixed 0–0.6 range. Grey, beige and blue indicate ignored, non-water and water pixels.

Export probabilities and masks as georeferenced TIFFs with nodata=-1. Model inputs retain the experiment's fixed preprocessing.

In [ ]:
SHOW_INDEX = 0
x4, target = FloodDataset(splits["test"], 4)[SHOW_INDEX]
image_name = splits["test"][SHOW_INDEX][0].replace("_S1Hand", "_S2Hand")
with rasterio.open(HAND / "S2Hand" / image_name) as src:
    profile = src.profile.copy()
valid = target.numpy() != -1
predictions = {}
for name, bands in EXPERIMENTS.items():
    model, checkpoint = load_best(name)
    with torch.inference_mode():
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x4[:len(bands)].unsqueeze(0).to(device))
        probability = logits.float().softmax(dim=1)[0, 1].cpu().numpy()
        mask = logits.argmax(dim=1)[0].cpu().numpy().astype("int16")
    mask[~valid] = -1
    probability[~valid] = -1
    predictions[name] = mask
    for suffix, array, dtype in [("mask", mask, "int16"), ("probability", probability, "float32")]:
        output_profile = {**profile, "count": 1, "dtype": dtype, "nodata": -1, "compress": "deflate"}
        with rasterio.open(RUN / name / f"{Path(image_name).stem}_{suffix}.tif", "w", **output_profile) as dst:
            dst.write(array, 1)
    del model, checkpoint
rgb = np.moveaxis(x4[:3].numpy(), 0, -1)
lo, hi = np.percentile(rgb[valid], [2, 98], axis=0)
rgb = np.clip((rgb - lo) / np.maximum(hi - lo, 1e-6), 0, 1)
cmap = ListedColormap(["#d7dee7", "#eee8d5", "#167caf"])
norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], 3)
fig, axes = plt.subplots(1, 5, figsize=(17, 4), layout="constrained")
axes[0].imshow(rgb); axes[0].set_title("RGB")
axes[1].imshow(x4[3], cmap="gray", vmin=0, vmax=0.6); axes[1].set_title("NIR · B8")
for ax, values, title in zip(axes[2:], [target, predictions["rgb"], predictions["rgb_nir"]],
                            ["Human label", "RGB prediction", "RGB+NIR prediction"]):
    im = ax.imshow(values, cmap=cmap, norm=norm, interpolation="nearest")
    ax.set_title(title)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
bar = fig.colorbar(im, ax=list(axes[2:]), ticks=[-1, 0, 1], shrink=0.6)
bar.ax.set_yticklabels(["Ignored", "Non-water", "Water"])
fig.suptitle(Path(image_name).stem)
fig.savefig(RUN / "test_prediction_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)


## Outputs

The run directory contains configuration, checkpoints, epoch CSVs, learning curves, test metrics and the prediction example. Save the executed notebook alongside these files.

References: [U-Net](https://arxiv.org/abs/1505.04597) · [Sen1Floods11](https://github.com/cloudtostreet/Sen1Floods11).